# General Loan Data Pipeline

Start here for a new deal tape. The expected inputs are one origination-level dataset and one periodic performance dataset. The notebook keeps raw, standardized, merged, summary, and regression-ready DataFrames in memory so they can be opened in Data Wrangler.

## Data Shape

- `origination_raw`: one row per loan with original balance, origination date, FICO, product type, term, rate, and other loan characteristics.
- `performance_raw`: one row per loan per reporting period with EOM date, current balance, delinquency, charge-off, recovery, and payment fields when available.
- `loan_period`: merged loan-period panel used for cohort review and regression preparation.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.6f}".format)

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from parentHelpers import format_table, plot_finance_style
except ImportError as exc:
    PARENT_HELPERS_IMPORT_ERROR = exc
    format_table = None
    plot_finance_style = None

from loanPipelineHelpers import *

try:
    from regression import run_model_pipeline
except ImportError as exc:
    REGRESSION_IMPORT_ERROR = exc
    run_model_pipeline = None

try:
    from itables import init_notebook_mode, options, show
    init_notebook_mode(all_interactive=True)
    options.pageLength = 25
    options.lengthMenu = [10, 25, 50, 75, 100]
    options.scrollX = True
except ImportError:
    show = display

## Configuration

Set file paths and map the raw tape columns into standard names. Leave optional fields blank until a tape has them.

In [ ]:
ORIGINATION_PATH = r""
PERFORMANCE_PATH = r""

ORIGINATION_READ_KWARGS = {}
PERFORMANCE_READ_KWARGS = {}

ORIGINATION_COLUMNS = {
    "loan_id": "",
    "origination_date": "",
    "original_balance": "",
    "fico": "",
    "product_type": "",
    "interest_rate": "",
    "term": "",
    "ltv": "",
    "dti": "",
}

PERFORMANCE_COLUMNS = {
    "loan_id": "",
    "performance_date": "",
    "current_balance": "",
    "days_delinquent": "",
    "chargeoff_amount": "",
    "recovery_amount": "",
    "scheduled_principal": "",
    "principal_payment": "",
}

REQUIRED_ORIGINATION_COLS = ["loan_id", "origination_date", "original_balance"]
REQUIRED_PERFORMANCE_COLS = ["loan_id", "performance_date", "current_balance"]

ORIGINATION_NUMERIC_COLS = ["original_balance", "fico", "interest_rate", "term", "ltv", "dti"]
PERFORMANCE_NUMERIC_COLS = ["current_balance", "days_delinquent", "chargeoff_amount", "recovery_amount", "scheduled_principal", "principal_payment"]

VINTAGE_FREQ = "Q"

In [ ]:
origination_raw = read_tabular(ORIGINATION_PATH, **ORIGINATION_READ_KWARGS) if ORIGINATION_PATH else pd.DataFrame()
performance_raw = read_tabular(PERFORMANCE_PATH, **PERFORMANCE_READ_KWARGS) if PERFORMANCE_PATH else pd.DataFrame()

print(f"origination_raw shape: {origination_raw.shape}")
print(f"performance_raw shape: {performance_raw.shape}")

show(origination_raw.head(50))
show(performance_raw.head(50))

## Standardize And Preview

After this cell, open `origination`, `performance`, or later `loan_period` in Data Wrangler for interactive cleaning. If Data Wrangler creates transformation code, paste it below this section so the pipeline stays reproducible.

In [ ]:
origination = standardize_columns(origination_raw, ORIGINATION_COLUMNS)
performance = standardize_columns(performance_raw, PERFORMANCE_COLUMNS)

origination = coerce_datetime(origination, ["origination_date"])
performance = coerce_datetime(performance, ["performance_date"])

origination = coerce_numeric(origination, ORIGINATION_NUMERIC_COLS)
performance = coerce_numeric(performance, PERFORMANCE_NUMERIC_COLS)

if "fico" in origination.columns:
    origination = add_fico_bucket(origination, fico_col="fico")

missing_origination_cols = missing_columns(origination, REQUIRED_ORIGINATION_COLS)
missing_performance_cols = missing_columns(performance, REQUIRED_PERFORMANCE_COLS)

print("Missing origination columns:", missing_origination_cols)
print("Missing performance columns:", missing_performance_cols)

show(origination.head(50))
show(performance.head(50))

## Data Quality Review

In [ ]:
origination_quality = data_quality_report(
    origination,
    required_cols=REQUIRED_ORIGINATION_COLS,
    weight_col="original_balance",
)

performance_quality = data_quality_report(
    performance,
    required_cols=REQUIRED_PERFORMANCE_COLS,
    weight_col="current_balance",
)

merge_coverage = (
    merge_coverage_report(origination, performance, loan_id_col="loan_id")
    if not origination.empty and not performance.empty and "loan_id" in origination.columns and "loan_id" in performance.columns
    else pd.DataFrame()
)

show(origination_quality)
show(performance_quality)
show(merge_coverage)

## Origination Statistics

In [ ]:
origination_stats_base = origination.copy()
if "origination_date" in origination_stats_base.columns:
    origination_stats_base["vintage"] = origination_stats_base["origination_date"].dt.to_period(VINTAGE_FREQ).astype("string")

origination_summary = weighted_summary(
    origination_stats_base,
    weight_col="original_balance",
    numeric_cols=["fico", "interest_rate", "term", "ltv", "dti"],
)

origination_by_vintage = weighted_summary(
    origination_stats_base,
    groupby_cols=["vintage"] if "vintage" in origination_stats_base.columns else None,
    weight_col="original_balance",
    numeric_cols=["fico", "interest_rate", "term", "ltv", "dti"],
)

fico_distribution = (
    weighted_distribution(origination_stats_base, "fico_bucket", weight_col="original_balance", by_col="vintage")
    if "fico_bucket" in origination_stats_base.columns and "vintage" in origination_stats_base.columns
    else pd.DataFrame()
)

product_distribution = (
    weighted_distribution(origination_stats_base, "product_type", weight_col="original_balance", by_col="vintage")
    if "product_type" in origination_stats_base.columns and "vintage" in origination_stats_base.columns
    else pd.DataFrame()
)

show(origination_summary)
show(origination_by_vintage)
show(fico_distribution)
show(product_distribution)

## Performance Panel And Cohort Review

In [ ]:
if not missing_origination_cols and not missing_performance_cols and not origination.empty and not performance.empty:
    loan_period = merge_origination_performance(
        origination,
        performance,
        loan_id_col="loan_id",
        how="left",
    )
    loan_period = add_vintage_and_mob(
        loan_period,
        origination_date_col="origination_date",
        performance_date_col="performance_date",
        vintage_freq=VINTAGE_FREQ,
    )
    loan_period_rollup = performance_rollup(
        loan_period,
        chargeoff_amount_col="chargeoff_amount",
        recovery_amount_col="recovery_amount",
    )
else:
    loan_period = pd.DataFrame()
    loan_period_rollup = pd.DataFrame()

print(f"loan_period shape: {loan_period.shape}")
show(loan_period.head(100))
show(loan_period_rollup.head(100))

## Data Wrangler Staging

These variables are intentionally simple copies. Use them for interactive inspection and transformations without losing the original pipeline objects.

In [ ]:
data_wrangler_origination = origination.copy()
data_wrangler_performance = performance.copy()
data_wrangler_loan_period = loan_period.copy()

for name, df in {
    "data_wrangler_origination": data_wrangler_origination,
    "data_wrangler_performance": data_wrangler_performance,
    "data_wrangler_loan_period": data_wrangler_loan_period,
}.items():
    print(f"{name}: {df.shape}")

## Regression Setup

Choose a target and feature list after the data review is complete. The default object is `loan_period`, but `origination` can be used for origination-only models.

In [ ]:
REGRESSION_DF = loan_period.copy()

MODEL_TARGET_COL = ""
CATEGORICAL_FEATURES = ["product_type", "fico_bucket"]
NUMERIC_FEATURES = ["fico", "interest_rate", "term", "ltv", "dti", "mob"]
SPLINE_FEATURES = None
MODEL_TYPE = "linear"

available_categorical_features = [col for col in CATEGORICAL_FEATURES if col in REGRESSION_DF.columns]
available_numeric_features = [col for col in NUMERIC_FEATURES if col in REGRESSION_DF.columns]

if run_model_pipeline is None:
    regression_results = None
    print(f"regression.py could not be imported: {REGRESSION_IMPORT_ERROR}")
elif MODEL_TARGET_COL and MODEL_TARGET_COL in REGRESSION_DF.columns and not REGRESSION_DF.empty:
    regression_results = run_model_pipeline(
        df=REGRESSION_DF,
        target_col=MODEL_TARGET_COL,
        categorical_features=available_categorical_features,
        numeric_features=available_numeric_features,
        spline_features=SPLINE_FEATURES,
        model_type=MODEL_TYPE,
        test_size=0.2,
        id_col="loan_id" if "loan_id" in REGRESSION_DF.columns else None,
        plot=True,
    )
else:
    regression_results = None
    print("Set MODEL_TARGET_COL after the data review section, then rerun this cell.")

print("Categorical features:", available_categorical_features)
print("Numeric features:", available_numeric_features)

## Post-Regression Review

After reviewing `regression_coef_df`, add only the useful drivers to `POST_REGRESSION_CHARTS`. Each chart shows actual vs model results by cohort with one selected driver on the secondary axis.

In [ ]:
regression_coef_df = (
    regression_results["coef_df"].copy()
    if regression_results is not None and "coef_df" in regression_results
    else pd.DataFrame()
)

show(regression_coef_df)

POST_REGRESSION_GROUP_COL = "vintage"
POST_REGRESSION_WEIGHT_COL = "original_balance"

POST_REGRESSION_CHARTS = [
    # Add charts after reviewing regression_coef_df, for example:
    # {"label": "FICO", "column": "fico", "kind": "weighted_average"},
    # {"label": "LTV", "column": "ltv", "kind": "weighted_average"},
    # {"label": "% Product A", "column": "product_type", "kind": "category_mix", "category": "Product A"},
]

if regression_results is not None and POST_REGRESSION_GROUP_COL in REGRESSION_DF.columns:
    post_regression_review_table = build_post_regression_review_table(
        regression_results,
        reference_df=REGRESSION_DF,
        group_col=POST_REGRESSION_GROUP_COL,
        weight_col=POST_REGRESSION_WEIGHT_COL,
        chart_configs=POST_REGRESSION_CHARTS,
        id_col="loan_id" if "loan_id" in REGRESSION_DF.columns else None,
        factor_reference_df=REGRESSION_DF,
    )
else:
    post_regression_review_table = pd.DataFrame()

show(post_regression_review_table)

if not post_regression_review_table.empty and POST_REGRESSION_CHARTS and plot_finance_style is not None:
    post_regression_chart_tables = plot_post_regression_review_charts(
        post_regression_review_table,
        chart_configs=POST_REGRESSION_CHARTS,
        plot_func=plot_finance_style,
        outcome_label=MODEL_TARGET_COL,
    )
else:
    post_regression_chart_tables = {}
    print("No post-regression charts selected yet, or plotting is unavailable.")

## Optional Output

In [ ]:
WRITE_OUTPUTS = False
OUTPUT_PATH = Path("pipeline_outputs.xlsx")

if WRITE_OUTPUTS:
    outputs = {
        "origination_summary": origination_summary,
        "origination_by_vintage": origination_by_vintage,
        "fico_distribution": fico_distribution,
        "product_distribution": product_distribution,
        "merge_coverage": merge_coverage,
        "loan_period_rollup": loan_period_rollup,
    }
    with pd.ExcelWriter(OUTPUT_PATH) as writer:
        for sheet_name, df in outputs.items():
            if df is not None and not df.empty:
                df.to_excel(writer, sheet_name=sheet_name[:31])
    print(f"Wrote {OUTPUT_PATH}")